# Transformer Decoder-Only モデルを使用したテキスト生成

## 概要

この例では、KerasNLP を使用して Transformer デコーダによる小型の生成モデルを構築します。生成モデルを使用すると、プロンプトから高度なテキストを生成できます。このラボでは、Transformer スタックのデコーダのみを使用するモデルを構築します。各ステージにおいて、特定の単語に対するアテンション レイヤは、文中でその前にある単語にのみアクセスできます。このようなモデルは、多くの場合、自己回帰（auto-regressive）モデルと呼ばれます。Decoder-only モデルの例としては、[Transformer XL](https://arxiv.org/pdf/1901.02860.pdf)、[GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)、[CTRL](https://arxiv.org/pdf/1909.05858.pdf) などがあります。

このモデルは、いくつかの小説から作成されたデータセットである [simplebooks-92](https://arxiv.org/abs/1911.12391) コーパスでトレーニングします。このデータセットは 1,573 冊の Gutenberg 書籍から作成されました。語彙サイズが小さく、単語の出現頻度が高いため、パラメータ数の少ない生成モデルをトレーニングする際に有利であり、このサンプルのようなタスクに適したデータセットです。

このノートブックでは、KerasNLP のトークン化、レイヤ、評価指標を使用してトレーニング プロセスを簡素化する方法を示し、KerasNLP のサンプリング ユーティリティを使用して出力テキストを生成する方法を説明します。

### 学習目標
- WordPiece トークナイザを使用して生成モデル用のデータセットを準備・トレーニングする方法を学ぶ
- Keras NLP を使用して生成モデルを構築する方法を学ぶ
- プロンプトからテキストを出力するためのさまざまな推論手法を学ぶ

## セットアップ

このノートブックを実行するには、`keras_hub` が必要です。Keras Hub は、シンプル、柔軟、かつ高速を目指した事前学習済みモデリング ライブラリです。TensorFlow、JAX、PyTorch とネイティブに連携します。Keras NLP は、このノートブックで生成モデルを構築するのに非常に役立つ Transformer レイヤを提供します。

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
import keras
import keras_hub
import tensorflow as tf

### 始める前に
トレーニングに時間がかかりすぎないように、Notebook インスタンスに GPU（1 x NVIDIA Tesla T4 で十分です）がアタッチされていることを確認してください。

GPU がアタッチされているかどうかを確認するには、次のコマンドを実行します。

In [ ]:
# GPU が 1 基接続されている場合、"Num GPUs Available: 1" と出力されるはずです
print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

## 設定とハイパーパラメータ

ここではいくつかのハイパーパラメータを初期化します。ここでの `SEQ_LEN` はデータセット内の文の最大長を定義し、これを `WordPieceTokenizer` の長さとして使用します。また、短すぎる文をデータセットからクリーンアップするために `MIN_TRAINING_SEQ_LEN` も定義します。

In [ ]:
# データ
BATCH_SIZE = 64
SEQ_LEN = 128
MIN_TRAINING_SEQ_LEN = 450

# モデル
EMBED_DIM = 256
FEED_FORWARD_DIM = 256
NUM_HEADS = 3
NUM_LAYERS = 2
VOCAB_SIZE = 5000  # モデルのパラメータ数を制限

## データの読み込み

このノートブックでは SimpleBooks データセットを使用します。SimpleBooks データセットは、1,573 冊の Gutenberg 書籍で構成され、単語レベルのトークンに対する語彙サイズ比が小さくなっています。語彙サイズは約 98k です。このサイズにより、小型の Transformer モデルに適合させやすくなります。このブロックでは、[TensorFlow の data API](https://www.tensorflow.org/api_docs/python/tf/data/TextLineDataset) を使用してデータを読み込みます。

**演習:** `tf.data.TextLineDataset` を使用して検証データを `raw_val_ds` に読み込みます。検証データへのパスは `simplebooks-92-raw/valid.txt` です。

In [ ]:
keras.utils.get_file(
    origin="https://storage.googleapis.com/asl-public/text/data/simplebooks.zip",
    extract=True,
)
data_dir = os.path.expanduser("~/.keras/datasets/simplebooks.zip/simplebooks/")

# simplebooks-92 トレーニング セットを読み込み、MIN_TRAINING_SEQ_LEN を使用して短い行を除外
raw_train_ds = (
    tf.data.TextLineDataset(data_dir + "simplebooks-92-raw/train.txt")
    .filter(lambda x: tf.strings.length(x) > MIN_TRAINING_SEQ_LEN)
    .batch(BATCH_SIZE)
    .shuffle(buffer_size=256)
)

# simplebooks-92 検証セットを読み込み、MIN_TRAINING_SEQ_LEN を使用して短い行を除外
raw_val_ds = (
    # TODO: トレーニング データと同様に、テキスト ファイルから検証データを読み込むコードを作成してください
)

## トークナイザのトレーニング

チューニングされたハイパーパラメータである語彙サイズ `VOCAB_SIZE` に合わせて、トレーニング データセットから Keras NLP の [compute_word_piece_vocabulary](https://keras.io/api/keras_hub/tokenizers/compute_word_piece_vocabulary/) を使用してトークナイザをトレーニングします。モデルのパラメータ数に大きな影響を与えるため、語彙は可能な限り制限したいと考えています。同時に、単語数が*少なすぎる*と語彙外（OOV: out-of-vocabulary）のサブワードが多すぎるため、それも避けたいところです。さらに、語彙内には 3 つのトークンが予約されています。

- `"[PAD]"`: シーケンスを `SEQ_LEN` までパディングするためのトークン。`WordPieceTokenizer`（および他のレイヤ）は `0`/`vocab[0]` をデフォルトのパディングと見なすため、このトークンは `reserved_tokens` と `vocab` の両方でインデックス 0 を持ちます。
- `"[UNK]"`: OOV サブワード用トークン。`WordPieceTokenizer` のデフォルトの `oov_token="[UNK]"` と一致する必要があります。
- `"[BOS]"`: 文の先頭（beginning of sentence）を表しますが、ここでは技術的にはトレーニング データの各行の先頭を表すトークンです。

データセット全体に対して WordPiece 語彙を計算しているため、このセルの実行には約 5〜10 分かかります。

In [ ]:
# トークナイザの語彙をトレーニング
print("Training the word piece tokenizer. This will take 5-10 mins...")
vocab = keras_hub.tokenizers.compute_word_piece_vocabulary(
    raw_train_ds,
    vocabulary_size=VOCAB_SIZE,
    lowercase=True,
    reserved_tokens=["[PAD]", "[UNK]", "[BOS]"],
)
print("Training is complete!!")

## トークナイザの読み込み

語彙データを使用して [keras_hub.tokenizers.WordPieceTokenizer](https://keras.io/api/keras_hub/tokenizers/word_piece_tokenizer/) を初期化します。WordPieceTokenizer は、BERT やその他のモデルで使用されている WordPiece アルゴリズムの効率的な実装です。空白の削除、小文字化、その他の不可逆的な前処理操作を実行します。語彙と入力文が与えられると、WordPiece トークナイザは文を ID の配列に変換し、定義された `SEQ_LEN` まで文をパディングします。たとえば、

```
vocab = ["[UNK]", "the", "qu", "##ick", "br", "##own", "fox", "."]
inputs = "The quick brown fox."
SEQ_LEN = 10
```

これを `WordPieceTokenizer` に渡すと、次のように返されます。
```
array([1, 2, 3, 4, 5, 6, 7,0,0,0], dtype=int32)
```

**演習:** `keras_hub.tokenizers.WordPieceTokenizer` の引数（vocabulary、sequence_length、lowercase）を定義します。[こちらのドキュメント](https://keras.io/api/keras_hub/tokenizers/word_piece_tokenizer/) を参照してください。

In [ ]:
tokenizer = keras_hub.tokenizers.WordPieceTokenizer(
    # TODO: 引数を入力してください
)

## データのトークン化

データセットをトークン化し、`features` と `labels` に分割して前処理します。これは言語モデリング タスクであるため、目標は「前の単語」の「特徴量シーケンス（features sequence）」から「次の単語」の「ラベル シーケンス（label sequence）」を予測することです。「特徴量」を取得するために、`[BOS]` トークンを使用して元の文を右にシフトします。

In [ ]:
# packer が開始トークンを追加します
start_packer = keras_hub.layers.StartEndPacker(
    sequence_length=SEQ_LEN,
    start_value=tokenizer.token_to_id("[BOS]"),
)


def preprocess(inputs):
    outputs = tokenizer(inputs)
    features = start_packer(outputs)
    labels = outputs
    return features, labels


# トークン化し、トレーニング用シーケンスとラベル シーケンスに分割。
train_ds = raw_train_ds.map(
    preprocess, num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)
val_ds = raw_val_ds.map(
    preprocess, num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

## モデルの構築

次のレイヤを使用して、Transformer デコーダに基づく小型のテキスト生成モデルを作成します。

- 1 つの `keras_hub.layers.TokenAndPositionEmbedding` レイヤ: トークンの埋め込みとその位置の埋め込みを結合します。これは、固定の正弦波埋め込みの代わりに学習可能な位置埋め込みを作成するため、従来の Embedding レイヤとは異なります。
- ループを使用して作成された複数の `keras_hub.layers.TransformerDecoder` レイヤ。
- 最終的な 1 つの Dense 線形レイヤ。

**注:** このレイヤを構成するさまざまなコンポーネントを確認するには、このレイヤの[ソースコード](https://github.com/keras-team/keras-hub/blob/v0.23.0/keras_hub/src/layers/modeling/token_and_position_embedding.py#L11)を参照してください。

**演習:** 次のレイヤを使用してモデル コードを作成します。
- Embedding レイヤ: 引数 vocabulary_size、sequence_length、embedding_dim、mask_zero を指定して [keras_hub.layers.TokenAndPositionEmbedding](https://keras.io/api/keras_hub/modeling_layers/token_and_position_embedding/) を使用します。
- Transformer レイヤ: 引数 num_heads、intermediate_dim を指定して [keras_hub.layers.TransformerDecoder](https://keras.io/api/keras_hub/modeling_layers/transformer_decoder/) を使用します。変数 `NUM_LAYERS` を使用して、設定したい Transformer レイヤの数を自由に調整してみてください。

In [ ]:
inputs = keras.layers.Input(shape=(None,), dtype="int32")
# Embedding レイヤ
embedding_layer = # TODO: Embedding レイヤのコードを記述してください
x = embedding_layer(inputs)
# Transformer デコーダ レイヤ
for _ in range(NUM_LAYERS):
    decoder_layer = # TODO: Transformer デコーダ レイヤの定義を記述してください
    x = decoder_layer(x)  # 引数を 1 つだけ渡すとクロス アテンションがスキップされます
# 出力レイヤ
outputs = keras.layers.Dense(VOCAB_SIZE)(x)
model = keras.Model(inputs=inputs, outputs=outputs)

# 損失と評価指標の設定
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
perplexity = keras_hub.metrics.Perplexity(from_logits=True, mask_token_id=0)

# モデルのコンパイル
model.compile(optimizer="adam", loss=loss_fn, metrics=[perplexity])

モデルの概要（summary）を確認してみましょう。パラメータの大半は `token_and_position_embedding` と出力の `dense` レイヤに含まれています。
これは、語彙サイズ（`VOCAB_SIZE`）がモデルのサイズに大きな影響を与える一方で、Transformer デコーダ レイヤの数（`NUM_LAYERS`）はそれほど影響しないことを意味します。

In [ ]:
model.summary()

## トレーニング

モデルが準備できたので、`fit()` メソッドを使用してトレーニングしましょう。

**演習:** トレーニング データと検証データを指定して `model.fit()` 関数を定義します。この関数には、次の引数を定義して渡す必要があります。
- トレーニング データ
- 検証データ
- 変数 EPOCHS で定義されるエポック数

In [ ]:
EPOCHS = 1  # より良い結果を得るにはエポック数を増やしてください
print("Training started, this could take 4-10 mins per epoch with a T4 GPU...")
model.fit(
    # TODO: 引数を入力してください
)
print("Training is complete!!")

## 推論

トレーニング済みモデルを使用してテストを行い、そのパフォーマンスを測定できます。これを行うには、`"[BOS]"` トークンで始まる入力シーケンスをモデルのシードとして与え、ループ内で後続の各トークンに対する予測を行うことでモデルから段階的にサンプリングします。

まず、モデルの入力と同じ形状で、`"[BOS]"` トークンのみを含むプロンプトを作成しましょう。

In [ ]:
# "packer" レイヤが [BOS] トークンを追加してくれます。
prompt_tokens = start_packer(tokenizer([""]))
prompt_tokens

推論には `keras_hub.samplers` モジュールを使用します。これには、先ほどトレーニングしたモデルをラップするコールバック関数が必要です。このラッパーはモデルを呼び出し、生成中の現在のトークンのロジット予測を返します。

**注:** コールバックを定義する際に利用できる高度な機能が 2 つあります。1 つ目は、前の生成ステップで計算された状態の `cache` を取り込む機能で、生成を高速化するために使用できます。2 つ目は、生成された各トークンの最終的な Dense「隠れ状態」を出力する機能です。これは繰り返される隠れ状態にペナルティを与えることで繰り返しを回避する `keras_hub.samplers.ContrastiveSampler` で使用されます。どちらもオプションであるため、ここでは無視します。

In [ ]:
def next(prompt, cache, index):
    logits = model(prompt)[:, index - 1, :]
    # ここでは隠れ状態を無視します。コントラスト探索にのみ必要です。
    hidden_states = None
    return logits, hidden_states, cache

ラッパー関数の作成は、これらの関数を使用する上で最も複雑な部分です。作成が完了したので、まずは貪欲探索（Greedy search）から、さまざまなユーティリティをテストしてみましょう。

### 貪欲探索（Greedy search）

各タイムステップで最も確率の高いトークンを貪欲に選択します。言い換えると、モデル出力の argmax を取得します。

In [ ]:
sampler = keras_hub.samplers.GreedySampler()
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,  # [BOS] トークンの直後からサンプリングを開始します。
)
txt = tokenizer.detokenize(output_tokens)
print(f"Greedy search generated text: \n{txt}\n")

ご覧のとおり、貪欲探索はある程度意味の通るテキストから始まりますが、すぐに同じ言葉を繰り返すようになります。これはテキスト生成における一般的な問題であり、後ほど紹介する確率的テキスト生成ユーティリティによって解決できます。

### ビーム探索（Beam search）

大まかに言えば、ビーム探索は各タイムステップで最も確率の高い上位 `num_beams` 個のシーケンスを追跡し、すべてのシーケンスから最適な次のトークンを予測します。より多くの可能性を保持するため、貪欲探索よりも改善されています。ただし、複数の潜在的なシーケンスを計算して保持する必要があるため、貪欲探索よりも効率は低下します。

**注:** `num_beams=1` のビーム探索は貪欲探索と同じです。

In [ ]:
sampler = keras_hub.samplers.BeamSampler(num_beams=10)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Beam search generated text: \n{txt}\n")

貪欲探索と同様に、ビーム探索も決定論的な手法であるため、すぐに同じ言葉を繰り返すようになります。

### ランダム探索（Random search）

ランダム探索は最初の確率的手法です。各タイムステップで、モデルから提供された softmax 確率を使用して次のトークンをサンプリングします。

In [ ]:
sampler = keras_hub.samplers.RandomSampler()
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Random search generated text: \n{txt}\n")

見事に繰り返しがなくなりました。ただし、ランダム探索では語彙内のすべての単語が出現する可能性があるため、意味をなさない単語が出現することがあります。これは、次の探索ユーティリティである Top-K 探索によって解決されます。

### Top-K 探索

ランダム探索と同様に、モデルから提供された確率分布から次のトークンをサンプリングします。唯一の違いは、最も確率の高い上位 `k` 個のトークンを選択し、サンプリング前にそれらのトークン全体に確率質量を再配分することです。これにより、確率の低いトークンからサンプリングされることがなくなるため、意味をなさない単語が少なくなります。

In [ ]:
sampler = keras_hub.samplers.TopKSampler(k=10)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Top-K search generated text: \n{txt}\n")

### Top-P 探索

Top-K 探索でも、まだ改善の余地があります。Top-K 探索では数値 `k` が固定されているため、どのような確率分布であっても同じ数のトークンが選択されます。確率質量が 2 単語に集中しているシナリオと、10 単語に均等に分散しているシナリオの 2 つを考えてみてください。`k=2` と `k=10` のどちらを選択すべきでしょうか。あらゆるケースに適した単一の `k` は存在しません。

そこで登場するのが Top-P 探索です。`k` を選択する代わりに、上位トークンの確率の合計として目標とする確率 `p` を選択します。これにより、確率分布に基づいて `k` を動的に調整できます。`p=0.9` に設定すると、確率質量の 90% が上位 2 つのトークンに集中している場合、サンプリング対象として上位 2 つのトークンのみが絞り込まれます。一方、90% が 10 個のトークンに分散している場合は、同様に上位 10 個のトークンが絞り込まれます。

In [ ]:
sampler = keras_hub.samplers.TopPSampler(p=0.5)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Top-P search generated text: \n{txt}\n")

### テキスト生成にコールバックを使用する

これらのユーティリティをコールバックでラップすることもできます。これにより、モデルのエポックごとに予測シーケンスを出力できます。これは、エポックごとにモデルが改善されているかどうかを確認するのに非常に便利です。以下は、Top-K 探索用のコールバックの例です。

In [ ]:
class TopKTextGenerator(keras.callbacks.Callback):
    """top-k を使用してトレーニング済みモデルからテキストを生成するコールバック。"""

    def __init__(self, k):
        self.sampler = keras_hub.samplers.TopKSampler(k)

    def on_epoch_end(self, epoch, logs=None):
        output_tokens = self.sampler(
            next=next,
            prompt=prompt_tokens,
            index=1,
        )
        txt = tokenizer.detokenize(output_tokens)
        print(f"Top-K search generated text: \n{txt}\n")


text_generation_callback = TopKTextGenerator(k=10)
# コールバックを実演するためのダミーのトレーニング ループ。
model.fit(
    train_ds.take(1), verbose=2, epochs=2, callbacks=[text_generation_callback]
)

## 謝辞
このノートブックは、[Jesse Chan による Keras チュートリアル](https://keras.io/examples/generative/text_generation_gpt/#train-the-tokenizer) に基づいています。Transformer デコーダ レイヤは、Google の研究論文 [Attention Is All You Need, Vaswani et al., 2017](https://arxiv.org/abs/1706.03762) に基づいています。

# License

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License